# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and available [here](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print main metadata
print(f"{dataset.metadata.name} (version {getattr(dataset.metadata, 'version', 'N/A')})")
print(f"Identifier: {getattr(dataset.metadata, 'identifier', 'N/A')}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields. All are referenced by their `@id` for reproducibility.

In [ ]:
# Helper to explore record sets, fields and columns via `@id`
from pprint import pprint

record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record set(s):\n")

for record_set in record_sets:
    print(f"- Record Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}) | Data type: {getattr(field, 'data_type', 'N/A')}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

> **Note:** All subsequent operations will reference record sets and fields by their `@id` values for clarity and stability.

In [ ]:
# Extract all records from each record set and load into DataFrames
dataframes = {}

for rs in record_sets:
    rs_id = rs.id
    # Load all records as list of dicts
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records from Record Set '{rs.name}' (@id: {rs_id})")
        print(f"Columns (field @id): {df.columns.tolist()}\n")
    else:
        print(f"No records found for Record Set '{rs.name}' (@id: {rs_id})\n")

# For further exploration, select the first non-empty record set
if dataframes:
    selected_record_set_id = next(iter(dataframes.keys()))  # take the first loaded
    df = dataframes[selected_record_set_id]
    print(f"Sample records from Record Set '@id': {selected_record_set_id}")
    display(df.head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.

Let's:  
* Filter records on a numeric field (e.g., 'MW' for molecular weight if present)  
* Normalize this field  
* Group by a text field (e.g., 'Description') if applicable


In [ ]:
import numpy as np

if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Attempt to auto-select an appropriate numeric field
    potential_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int64, np.int32] or pd.api.types.is_numeric_dtype(df[c])]
    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 0
        print(f"\nFiltering for {numeric_field} > {threshold:.2f}")
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field} above mean:")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first object-type column except numeric_field
        candidate_group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable field found for grouping.")
    else:
        print("No numeric field found to use for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and group means, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field} (@id: {numeric_field})")
    plt.show()

    if 'group_field' in locals():
        # Visualize group means
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.ylabel(f"Mean of {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we've looked at the FAIR^2 dataset package on human protein mass spec analysis with `mlcroissant`:

* Loaded the Croissant metadata and previewed dataset descriptive fields.
* Enumerated all record sets and fields by their stable `@id`.
* Loaded a record set into a DataFrame, and demonstrated EDA: filtering, normalization, and grouping on fields using only `@id` for referencing.
* Produced basic visualizations of key numeric fields as a starting point for further biological/proteomics analysis.

For more details and field definitions, refer to the [dataset Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

To cite this dataset:
> Kulka, M., Rodriguez Miera, S., and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.